In [1]:
import pandas as pd

DATA_DIR = r"C:/Users/刘丹琳/Desktop/IDX"

sold_input = f"{DATA_DIR}\\Sold_aggregate_cleaned.csv"
listings_input = f"{DATA_DIR}\\Listings_aggregate_cleaned.csv"

sold_output = f"{DATA_DIR}\\Sold_aggregate_WithRates.csv"
listings_output = f"{DATA_DIR}\\Listings_aggregate_WithRates.csv"

In [2]:
sold = pd.read_csv(sold_input)
listings = pd.read_csv(listings_input)
print(f"Sold loaded: {len(sold):,} rows")
print(f"Listings loaded: {len(listings):,} rows")

C:\Users\刘丹琳\AppData\Local\Temp\ipykernel_26764\3332002295.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv(sold_input)
C:\Users\刘丹琳\AppData\Local\Temp\ipykernel_26764\3332002295.py:2: DtypeWarning: Columns (0: ListAgentEmail) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv(listings_input)


Sold loaded: 639,776 rows
Listings loaded: 929,795 rows


In [3]:
# CREATE year_month KEYS
sold['CloseDate_parsed'] = pd.to_datetime(sold['CloseDate'], errors='coerce', format='%Y-%m-%d')
mask = sold['CloseDate_parsed'].isna()
if mask.any():
    sold.loc[mask, 'CloseDate_parsed'] = pd.to_datetime(
        sold.loc[mask, 'CloseDate'], errors='coerce', format='%m/%d/%Y'
    )

sold['year_month'] = sold['CloseDate_parsed'].dt.to_period('M')
sold_valid = sold['year_month'].notna().sum()
print(f"Sold: {sold_valid:,} / {len(sold):,} valid CloseDate records")


listings['ListingDate_parsed'] = pd.to_datetime(listings['ListingContractDate'], errors='coerce', format='%Y-%m-%d')
mask_l = listings['ListingDate_parsed'].isna()
if mask_l.any():
    listings.loc[mask_l, 'ListingDate_parsed'] = pd.to_datetime(
        listings.loc[mask_l, 'ListingContractDate'], errors='coerce', format='%m/%d/%Y'
    )

listings['year_month'] = listings['ListingDate_parsed'].dt.to_period('M')
listings_valid = listings['year_month'].notna().sum()
print(f"Listings: {listings_valid:,} / {len(listings):,} valid ListingContractDate records")

Sold: 639,776 / 639,776 valid CloseDate records
Listings: 929,795 / 929,795 valid ListingContractDate records


In [4]:
# FETCH MORTGAGE RATES FROM FRED
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US" 
mortgage = pd.read_csv(url, parse_dates=['observation_date']) 
mortgage.columns = ['date', 'rate_30yr_fixed'] 
print(f"Downloaded {len(mortgage):,} weekly records")
print(f"Date range: {mortgage['date'].min()} to {mortgage['date'].max()}")

Downloaded 2,885 weekly records
Date range: 1971-04-02 00:00:00 to 2026-07-09 00:00:00


In [5]:
# STEP 4: Resampling to monthly averages
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index()
print(f"Resampled to {len(mortgage_monthly):,} monthly averages")
print(mortgage_monthly.head())

Resampled to 664 monthly averages
  year_month  rate_30yr_fixed
0    1971-04           7.3100
1    1971-05           7.4250
2    1971-06           7.5300
3    1971-07           7.6040
4    1971-08           7.6975


In [6]:
# MERGE
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
print(f"Shape: {sold_with_rates.shape}")
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')
print(f"Shape: {listings_with_rates.shape}")


Shape: (639776, 87)
Shape: (929795, 46)


In [7]:
# VALIDATION
sold_null = sold_with_rates['rate_30yr_fixed'].isna().sum()
if sold_null == 0:
    print("  Sold: ✓ VALIDATION PASSED - No null rates")
else:
    print(f"  Sold: ⚠ {sold_null} null rates found")
    missing = sold_with_rates[sold_with_rates['rate_30yr_fixed'].isna()]['year_month'].unique()
    print(f"    Missing months: {missing[:10]}")

listings_null = listings_with_rates['rate_30yr_fixed'].isna().sum()
if listings_null == 0:
    print("  Listings: VALIDATION PASSED - No null rates")
else:
    print(f"  Listings: {listings_null} null rates found")
    missing = listings_with_rates[listings_with_rates['rate_30yr_fixed'].isna()]['year_month'].unique()
    print(f"    Missing months: {missing[:10]}")

  Sold: ✓ VALIDATION PASSED - No null rates
  Listings: VALIDATION PASSED - No null rates


In [8]:
# SAVE
sold_with_rates.to_csv(sold_output, index=False)
listings_with_rates.to_csv(listings_output, index=False)

print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())
print(listings_with_rates[['ListingContractDate', 'year_month', 'ListPrice', 'rate_30yr_fixed']].head())

    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-24    2024-01       950.0           6.6425
2  2024-01-16    2024-01     45000.0           6.6425
3  2024-01-08    2024-01    141500.0           6.6425
4  2024-01-17    2024-01     15000.0           6.6425
  ListingContractDate year_month  ListPrice  rate_30yr_fixed
0          2024-01-29    2024-01    90000.0           6.6425
1          2024-01-11    2024-01  1500000.0           6.6425
2          2024-01-01    2024-01  1340000.0           6.6425
3          2024-01-24    2024-01  2500000.0           6.6425
4          2024-01-12    2024-01  3150000.0           6.6425
